# Do students cluster? PCA, t-SNE and UMAP on all variables

Every respondent, every variable, no grouping imposed in advance. Question:

> Projected into two dimensions, do the survey responses fall into recognisable groups
> of students -- and is any apparent grouping more than noise?

**Read this before reading any plot below.** The embeddings are built to be checked, not
just looked at, and the checking is the point:

- **t-SNE and UMAP will show clusters whether or not clusters exist.** Both optimise
  local neighbourhood preservation, so at n=49 they reliably break a structureless cloud
  into tidy-looking blobs. Finding "crazy clustering" in a t-SNE plot is not evidence of
  anything by itself. This is the single most common way these methods get misused.
- Therefore every embedding here is run **twice**: once on the real data, once on a
  column-permuted copy that has identical marginal distributions and *zero* association
  between variables. If the two panels look equally clustered, the structure is an
  artefact of the method.
- Section 6 turns that into a number: silhouette score of k-means on the real data
  against its distribution over 200 permuted datasets. That gives a defensible answer to
  "is there structure", which no amount of staring at a scatter plot can.

The honest expected outcome is **no structure beyond noise** -- and that is a reportable
finding, not a failed analysis. Written up as *"we looked for subgroups using three
dimension-reduction methods and found none distinguishable from a permuted null"*, it is
worth more than a cherry-picked blob.

## Why this is hard with these data, and what I changed

I built what was asked for, with four deliberate modifications. Each is a place the
plain version would have produced something misleading.

**1. p > n.** Using every variable gives ~45-55 columns (18 course dummies, 11 area
dummies, ~9 domain dummies, 6 proficiency/comfort, plus type/language/research) on
**49 respondents** -- fewer students than variables. Consequences: PCA can fit at most 48
components and PC1+PC2 will explain a small share of variance (Section 4 prints it --
if it is under ~25%, the 2D picture is not a faithful summary of the data); and in
~50 binary dimensions all pairwise distances converge toward each other, so the
"nearest neighbours" that t-SNE and UMAP depend on become close to arbitrary. Mitigation:
run three nested feature sets (all / background-only / interests-only) rather than only
the kitchen sink, and drop near-constant columns.

**2. Near-constant columns carry no information but do add noise.** PSTAT 126 is 50/51,
PSTAT 120 49/51. After standardising, a column where one person differs becomes a single
huge z-score -- i.e. one student's quirk gets the same weight as a real 50/50 split, and
will often *drive* PC1. Dropped at a 5%/95% prevalence threshold.

**3. Mixed types need a scaling decision, and it changes the answer.** Comfort is 2-5,
proficiency is 1-3, course dummies are 0/1. Unscaled, comfort dominates by range;
standardised, rare binaries dominate by having tiny standard deviations. Neither is
neutral. Handled by running the standardised version as primary and a binary-only set
with Jaccard distance as a cross-check -- if the two disagree, the "clusters" are an
artefact of the scaling choice. (The instructor's `week2-surveys.R` clusters unscaled and
flags this exact question as an open one.)

**4. `areas` must not be split on commas.** Two option labels contain internal commas
("Predictive modeling, generally", "Statistical models and inference, generally"). Naive
splitting yields 74 phantom "generally" tokens and corrupts both labels. The canonical
11-label list is hard-coded below and matched whole.

**Also note:** joining background to interests drops to **49** respondents (those who
consented to both sections) from 51 and 49 respectively. Background-only runs keep 51.

In [ ]:
# Requires scikit-learn. UMAP is optional and degrades gracefully if absent:
#   pip install scikit-learn
#   pip install umap-learn      # optional, for Section 8 only
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

try:
    import umap
    HAVE_UMAP = True
except ImportError:
    HAVE_UMAP = False
    print('umap-learn not installed; Section 8 will be skipped. pip install umap-learn')

SEED = 197
rng = np.random.default_rng(SEED)

CANDIDATES = [Path('data'), Path('../data'), Path.cwd().parent / 'data']
DATA = next((p for p in CANDIDATES if p.is_dir()), None)
if DATA is None:
    raise FileNotFoundError('Could not locate data/. Run from the repo root or scripts/.')
print('Reading from:', DATA.resolve())

## 1. Encoding every variable

One row per respondent, one numeric column per variable. Choices made here, all of which
affect the embeddings:

| Variable | Encoding | Note |
|---|---|---|
| `prog/math/stat.comf` | as-is (2-5) | already numeric |
| `prog/math/stat.prof` | beg/int/adv -> 1/2/3 | imposes equal spacing; an assumption, not a measurement |
| `upper_div_courses` | 3-5 / 6-8 / 9+ -> 1/2/3 | same caveat |
| `domain_specialization`, `research` | one-hot | `Unsure` is a real third answer, not missing |
| `courses` | one 0/1 column per course | comma split is safe for this column |
| `type`, `language` | one-hot | |
| `domains` | 0/1 per canonical label, free text -> `Other` | |
| `areas` | 0/1 per canonical label, **whole-label match** | never split on commas |

In [ ]:
background = pd.read_csv(DATA / 'background-clean.csv').set_index('response_id')
interest = pd.read_csv(DATA / 'interest-clean.csv').set_index('response_id')

PROF_MAP = {'beg': 1, 'int': 2, 'adv': 3}
UPPER_MAP = {'3-5': 1, '6-8': 2, '9+': 3}

# Canonical option labels. Two of the areas labels contain commas; that is why they
# are matched as complete strings rather than produced by splitting.
AREA_LABELS = [
    'Analysis or classification of images',
    'Deep learning and neural networks',
    'Spatial statistics or time series analysis',
    'Data acquisition and engineering',
    'Natural language processing and analysis of text',
    'Model deployment and software or web integrations',
    'Data visualization and interactive dashboards',
    'Predictive modeling, generally',
    'Statistical models and inference, generally',
    'Databases',
    'Algorithms',
]
DOMAIN_LABELS = [
    'Technology', 'Software development', 'Neuroscience', 'Public health',
    'Biology', 'Environmental science', 'Social or political science', 'Ecology',
]


def dummies_from_split(series, prefix):
    """0/1 columns from a comma-separated multi-select. Safe only when no label
    contains a comma -- true for `courses`, NOT true for `areas`."""
    exploded = (
        series.str.split(',').explode().str.strip().replace('', np.nan).dropna()
    )
    return pd.crosstab(exploded.index, exploded).clip(upper=1).add_prefix(prefix)


def dummies_from_labels(series, labels, prefix, other_label=None):
    """0/1 columns by matching complete option labels. Immune to internal commas."""
    filled = series.fillna('')
    out = pd.DataFrame(
        {prefix + lab: filled.str.contains(lab, regex=False).astype(int) for lab in labels},
        index=series.index,
    )
    if other_label is not None:
        # Anything left once the known labels are stripped out is free text.
        residual = filled
        for lab in labels:
            residual = residual.str.replace(lab, '', regex=False)
        out[prefix + other_label] = (
            residual.str.replace(r'[,\s]', '', regex=True).str.len() > 0
        ).astype(int)
    return out


# --- background block -------------------------------------------------------
bg = pd.DataFrame(index=background.index)
for col in ['prog.comf', 'math.comf', 'stat.comf']:
    bg[col] = background[col]
for col in ['prog.prof', 'math.prof', 'stat.prof']:
    bg[col] = background[col].map(PROF_MAP)
bg['upper_div'] = background['upper_div_courses'].map(UPPER_MAP)
bg = bg.join(pd.get_dummies(background['domain_specialization'], prefix='domspec').astype(int))
bg = bg.join(pd.get_dummies(background['research'], prefix='research').astype(int))
course_dummies = dummies_from_split(background['courses'], 'course:')
bg = bg.join(course_dummies).fillna(0)

# --- interest block ---------------------------------------------------------
itr = pd.DataFrame(index=interest.index)
itr = itr.join(pd.get_dummies(interest['type'], prefix='type').astype(int))
itr = itr.join(pd.get_dummies(interest['language'], prefix='lang').astype(int))
itr = itr.join(dummies_from_labels(interest['domains'], DOMAIN_LABELS, 'domain:', 'Other'))
itr = itr.join(dummies_from_labels(interest['areas'], AREA_LABELS, 'area:'))

assert bg.isna().sum().sum() == 0, 'unexpected NAs in background block'
assert itr.isna().sum().sum() == 0, 'unexpected NAs in interest block'

# Sanity check the comma-safe encoding against the known counts.
print('area: Predictive modeling, generally ->',
      int(itr['area:Predictive modeling, generally'].sum()), '(expect 43)')
print('area: Statistical models and inference, generally ->',
      int(itr['area:Statistical models and inference, generally'].sum()), '(expect 31)')
print()
print('background block:', bg.shape, ' interest block:', itr.shape)

In [ ]:
# Inner join = the 49 respondents who consented to both sections.
full = bg.join(itr, how='inner')
print('joined (both consents):', full.shape)

MIN_PREV, MAX_PREV = 0.05, 0.95


def drop_near_constant(X, lo=MIN_PREV, hi=MAX_PREV, verbose=True):
    """Remove columns with almost no variation. For 0/1 columns the bounds are
    prevalence; for the ordinal columns they never bind."""
    binary = X.columns[X.isin([0, 1]).all()]
    prev = X[binary].mean()
    drop = prev[(prev < lo) | (prev > hi)].index.tolist()
    drop += [c for c in X.columns if X[c].nunique() <= 1 and c not in drop]
    if verbose and drop:
        print(f'  dropped {len(drop)} near-constant columns:',
              ', '.join(sorted(drop)))
    return X.drop(columns=drop)


FEATURE_SETS = {}
for name, block in [('all', full),
                    ('background', bg),
                    ('interests', itr)]:
    print(f'{name}: {block.shape[1]} raw columns, n={len(block)}')
    FEATURE_SETS[name] = drop_near_constant(block)
    print(f'  -> {FEATURE_SETS[name].shape[1]} retained\n')

# Binary-only cross-check set (courses + domains + areas), kept unscaled for Jaccard.
binary_cols = [c for c in FEATURE_SETS['all'].columns
               if c.startswith(('course:', 'domain:', 'area:'))]
FEATURE_SETS['binary_only'] = FEATURE_SETS['all'][binary_cols]
print('binary_only:', FEATURE_SETS['binary_only'].shape)

PRIMARY = 'all'

## 2. The null baseline

Shuffle each column independently. This keeps every marginal distribution exactly as
observed -- same share of Python preferences, same course prevalences, same comfort
distribution -- while destroying all association *between* variables. So a permuted
dataset is, by construction, a class of students with no subgroups in it.

Any visual or numeric structure that shows up just as strongly in the permuted data is
produced by the method and the sample size, not by the students.

In [ ]:
def permute_columns(X, generator):
    """Independently shuffle every column: preserves marginals, kills associations."""
    out = X.copy()
    for col in out.columns:
        out[col] = generator.permutation(out[col].to_numpy())
    return out


def standardize(X):
    return pd.DataFrame(
        StandardScaler().fit_transform(X), index=X.index, columns=X.columns
    )


X_real = standardize(FEATURE_SETS[PRIMARY])
X_null = standardize(permute_columns(FEATURE_SETS[PRIMARY], rng))

print('real:', X_real.shape, ' null:', X_null.shape)
print('marginals preserved?',
      np.allclose(sorted(FEATURE_SETS[PRIMARY].iloc[:, 0]),
                  sorted(permute_columns(FEATURE_SETS[PRIMARY], rng).iloc[:, 0])))

## 3. PCA -- and the variance-explained reality check

PCA first because it is the only one of the three that is linear, deterministic, and
interpretable: components are weighted sums of the original variables, so they can be
read. It is also the one the instructor's R script uses.

The number that decides whether the 2D scatter plots in this notebook mean anything is
**cumulative variance explained by PC1 + PC2**. With ~50 columns on 49 people, expect it
to be low. If it is ~20%, then 80% of what distinguishes these students is invisible in
the plot, and two points sitting together may be nothing alike.

In [ ]:
pca_real = PCA(random_state=SEED).fit(X_real)
pca_null = PCA(random_state=SEED).fit(X_null)

evr_real = pca_real.explained_variance_ratio_
evr_null = pca_null.explained_variance_ratio_

print(f'PC1+PC2 explain {evr_real[:2].sum():.1%} of variance (real)')
print(f'PC1+PC2 explain {evr_null[:2].sum():.1%} of variance (permuted null)')
print(f'components needed for 80% (real): {np.argmax(np.cumsum(evr_real) >= 0.8) + 1}')
print(f'components needed for 80% (null): {np.argmax(np.cumsum(evr_null) >= 0.8) + 1}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
k = min(20, len(evr_real))
axes[0].bar(np.arange(1, k + 1) - 0.2, evr_real[:k], width=0.4, label='real')
axes[0].bar(np.arange(1, k + 1) + 0.2, evr_null[:k], width=0.4, label='permuted null')
axes[0].set(xlabel='component', ylabel='variance explained', title='Scree: real vs null')
axes[0].legend()

axes[1].plot(np.arange(1, len(evr_real) + 1), np.cumsum(evr_real), marker='o', ms=3, label='real')
axes[1].plot(np.arange(1, len(evr_null) + 1), np.cumsum(evr_null), marker='o', ms=3, label='null')
axes[1].axhline(0.8, ls='--', c='grey', lw=1)
axes[1].set(xlabel='components', ylabel='cumulative variance', title='Cumulative variance')
axes[1].legend()
plt.tight_layout()
plt.show()

# A real signal shows up as early components lifting clearly above the null curve.
# Two curves lying on top of each other means the correlation structure is weak.

In [ ]:
# What do PC1 and PC2 actually mean? Largest-magnitude loadings on each.
loadings = pd.DataFrame(
    pca_real.components_[:3].T,
    index=X_real.columns,
    columns=['PC1', 'PC2', 'PC3'],
)

for pc in ['PC1', 'PC2']:
    top = loadings[pc].reindex(loadings[pc].abs().sort_values(ascending=False).index).head(10)
    print(f'--- {pc} ({evr_real[["PC1", "PC2"].index(pc)]:.1%} of variance): top loadings ---')
    print(top.round(3).to_string())
    print()

# Interpretability check: if PC1's top loadings are a grab-bag of unrelated rare
# binaries, it is capturing a few individuals rather than a dimension of the class.

In [ ]:
scores_real = pca_real.transform(X_real)[:, :2]
scores_null = pca_null.transform(X_null)[:, :2]

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5), sharex=True, sharey=True)
for ax, scores, title in [(axes[0], scores_real, 'Real data'),
                          (axes[1], scores_null, 'Permuted null')]:
    ax.scatter(scores[:, 0], scores[:, 1], s=45, alpha=0.75, edgecolor='white')
    ax.axhline(0, lw=0.5, c='grey')
    ax.axvline(0, lw=0.5, c='grey')
    ax.set(xlabel='PC1', ylabel='PC2', title=title)
fig.suptitle(f'PCA, feature set "{PRIMARY}" (n={len(X_real)}, '
             f'PC1+PC2 = {evr_real[:2].sum():.0%} of variance)')
plt.tight_layout()
plt.show()

In [ ]:
# Same PCA scores, coloured by variables we did not feed to the projection as a label.
# If an existing variable lines up with the layout, that is the interpretable story --
# much more useful for a write-up than an unlabelled blob.
COLOR_BY = {
    'language': interest['language'].reindex(X_real.index),
    'type': interest['type'].reindex(X_real.index),
    'research': background['research'].reindex(X_real.index),
    'stat.prof': background['stat.prof'].reindex(X_real.index),
    'upper_div_courses': background['upper_div_courses'].reindex(X_real.index),
    'domain_specialization': background['domain_specialization'].reindex(X_real.index),
}

fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=True, sharey=True)
for ax, (label, values) in zip(axes.ravel(), COLOR_BY.items()):
    for level in pd.unique(values.dropna()):
        mask = (values == level).to_numpy()
        ax.scatter(scores_real[mask, 0], scores_real[mask, 1], s=40, alpha=0.8,
                   edgecolor='white', label=f'{level} ({mask.sum()})')
    ax.set_title(label)
    ax.legend(fontsize=7, loc='best')
fig.suptitle('PCA scores coloured by observed variables (real data)')
plt.tight_layout()
plt.show()

## 4. t-SNE -- across perplexities, against the null

Two rules that make t-SNE readable rather than decorative:

1. **Never trust one perplexity.** It sets how many neighbours each point attends to;
   at n=49, perplexity 5 vs 25 is "5 neighbours" vs "half the class". Structure that only
   appears at one setting is a tuning artefact. Real structure persists across the row.
2. **Distances between blobs are meaningless.** t-SNE preserves neighbourhoods, not
   global geometry. Blob sizes and the gaps between them carry no information.

Top row is real, bottom row is the permuted null at matching settings. Compare *down*
the columns, not across.

In [ ]:
PERPLEXITIES = [5, 10, 15, 25]  # must stay below n


def run_tsne(X, perplexity, metric='euclidean'):
    return TSNE(
        n_components=2,
        perplexity=perplexity,
        metric=metric,
        init='pca' if metric == 'euclidean' else 'random',
        learning_rate='auto',
        random_state=SEED,
    ).fit_transform(X.to_numpy().astype(float))


fig, axes = plt.subplots(2, len(PERPLEXITIES), figsize=(4 * len(PERPLEXITIES), 8))
for j, perp in enumerate(PERPLEXITIES):
    for i, (X, tag) in enumerate([(X_real, 'real'), (X_null, 'null')]):
        emb = run_tsne(X, perp)
        axes[i, j].scatter(emb[:, 0], emb[:, 1], s=40, alpha=0.8, edgecolor='white',
                           color='tab:blue' if tag == 'real' else 'tab:grey')
        axes[i, j].set(title=f'{tag}, perplexity={perp}', xticks=[], yticks=[])
fig.suptitle('t-SNE: real (top) vs permuted null (bottom). Equally clustered = artefact.')
plt.tight_layout()
plt.show()

In [ ]:
# Cross-check on the binary-only set with Jaccard distance, which is built for
# presence/absence data and ignores jointly-absent pairs. Euclidean distance on 0/1
# columns treats "neither of us took PSTAT 175" as evidence of similarity; Jaccard does
# not. If the two give different pictures, the scaling choice is driving the result.
X_bin = FEATURE_SETS['binary_only']
X_bin_null = permute_columns(X_bin, rng)

fig, axes = plt.subplots(2, len(PERPLEXITIES), figsize=(4 * len(PERPLEXITIES), 8))
for j, perp in enumerate(PERPLEXITIES):
    for i, (X, tag) in enumerate([(X_bin, 'real'), (X_bin_null, 'null')]):
        emb = run_tsne(X, perp, metric='jaccard')
        axes[i, j].scatter(emb[:, 0], emb[:, 1], s=40, alpha=0.8, edgecolor='white',
                           color='tab:green' if tag == 'real' else 'tab:grey')
        axes[i, j].set(title=f'{tag}, perplexity={perp}', xticks=[], yticks=[])
fig.suptitle('t-SNE on binary variables only, Jaccard distance')
plt.tight_layout()
plt.show()

## 5. UMAP -- same discipline

UMAP keeps a little more global structure than t-SNE and is faster, but shares the core
hazard: it is *designed* to produce compact, well-separated groups, and it obliges even
when handed noise. `n_neighbors` plays the role perplexity does, and at n=49 anything
above ~25 is looking at over half the class.

Skipped automatically if `umap-learn` is not installed -- it is not needed for any
conclusion here, since PCA plus Section 6 already answer the question.

In [ ]:
N_NEIGHBORS = [5, 10, 15, 25]

if not HAVE_UMAP:
    print('Skipped: umap-learn not installed (pip install umap-learn).')
else:
    fig, axes = plt.subplots(2, len(N_NEIGHBORS), figsize=(4 * len(N_NEIGHBORS), 8))
    for j, nn in enumerate(N_NEIGHBORS):
        for i, (X, tag) in enumerate([(X_real, 'real'), (X_null, 'null')]):
            emb = umap.UMAP(
                n_neighbors=nn, min_dist=0.1, n_components=2, random_state=SEED,
            ).fit_transform(X.to_numpy().astype(float))
            axes[i, j].scatter(emb[:, 0], emb[:, 1], s=40, alpha=0.8, edgecolor='white',
                               color='tab:orange' if tag == 'real' else 'tab:grey')
            axes[i, j].set(title=f'{tag}, n_neighbors={nn}', xticks=[], yticks=[])
    fig.suptitle('UMAP: real (top) vs permuted null (bottom)')
    plt.tight_layout()
    plt.show()

## 6. The actual test: is there more cluster structure than noise?

Everything above is visual. This is the cell that produces a defensible answer.

For each k from 2 to 8: fit k-means, record the silhouette score (how well-separated the
clusters are, from -1 to 1), then repeat on 200 independently permuted datasets. The
permuted distribution is what silhouette looks like when there is *definitionally* no
structure. The real score's percentile against it is the result.

k-means always returns k clusters and silhouette is always positive-ish, so an absolute
score of 0.15 means nothing in isolation. **Only the comparison to the null interprets
it.** Rough reading: above the 95th percentile of the null is worth investigating;
inside the null's bulk means no detectable structure, which is a finding worth one
honest sentence in the report.

In [ ]:
K_RANGE = range(2, 9)
N_PERM = 200


def best_silhouette(X, k, seed=SEED):
    labels = KMeans(n_clusters=k, n_init=25, random_state=seed).fit_predict(X)
    if len(np.unique(labels)) < 2:
        return np.nan
    return silhouette_score(X, labels)


raw = FEATURE_SETS[PRIMARY]
rows = []
perm_rng = np.random.default_rng(SEED + 1)
null_draws = {k: [] for k in K_RANGE}

for _ in range(N_PERM):
    Xp = standardize(permute_columns(raw, perm_rng))
    for k in K_RANGE:
        null_draws[k].append(best_silhouette(Xp, k))

for k in K_RANGE:
    observed = best_silhouette(X_real, k)
    draws = np.array(null_draws[k])
    rows.append({
        'k': k,
        'silhouette_real': round(observed, 4),
        'null_mean': round(draws.mean(), 4),
        'null_p95': round(np.percentile(draws, 95), 4),
        'percentile_of_real': round(100 * (draws < observed).mean(), 1),
        'exceeds_null_p95': observed > np.percentile(draws, 95),
    })

sil = pd.DataFrame(rows)
print(f'k-means silhouette vs {N_PERM} column-permuted nulls, feature set "{PRIMARY}"\n')
print(sil.to_string(index=False))
print()
if not sil['exceeds_null_p95'].any():
    print('No k exceeds the 95th percentile of the null.')
    print('=> No cluster structure detectable above noise. Report this as the finding.')
else:
    hits = sil.loc[sil['exceeds_null_p95'], 'k'].tolist()
    print(f'k = {hits} exceed the null 95th percentile. Profile them in Section 7,')
    print('and check the same k on the "background" and "interests" sets before believing it.')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.boxplot([null_draws[k] for k in K_RANGE], positions=list(K_RANGE), widths=0.5)
ax.plot(sil['k'], sil['silhouette_real'], 'o-', color='crimson', label='real data')
ax.set(xlabel='k', ylabel='silhouette score',
       title=f'Real silhouette vs null distribution ({N_PERM} permutations)')
ax.legend()
plt.tight_layout()
plt.show()

# Red line inside the boxes = the clusters k-means found are no better separated than
# clusters found in data with every association deliberately destroyed.

In [ ]:
# Does the answer hold across feature sets? A conclusion that flips depending on which
# variables are included is not a conclusion. Smaller nulls here to keep it quick.
print(f'{"set":<12}{"n":>4}{"cols":>6}{"k":>4}{"real":>9}{"null_p95":>10}  exceeds')
for name in ['all', 'background', 'interests']:
    block = FEATURE_SETS[name]
    Xs = standardize(block)
    r = np.random.default_rng(SEED + 7)
    for k in [2, 3, 4]:
        observed = best_silhouette(Xs, k)
        draws = np.array([
            best_silhouette(standardize(permute_columns(block, r)), k) for _ in range(50)
        ])
        p95 = np.percentile(draws, 95)
        print(f'{name:<12}{len(block):>4}{block.shape[1]:>6}{k:>4}'
              f'{observed:>9.3f}{p95:>10.3f}  {observed > p95}')

## 7. Profiling clusters -- only if Section 6 justified it

If nothing cleared the null, skip this and write up the negative result. If something
did, a cluster is only reportable once it can be *described*: "cluster 2 is the
Python-preferring, deep-learning-interested group with more CS coursework" is a finding;
"cluster 2" is not.

This prints the variables where clusters differ most, on the original scales. Cluster
numbers are arbitrary labels -- interpret the centres, never the numbering.

In [ ]:
K_PROFILE = 3  # set to whatever Section 6 supported

labels = KMeans(n_clusters=K_PROFILE, n_init=25, random_state=SEED).fit_predict(X_real)
print('cluster sizes:', pd.Series(labels).value_counts().sort_index().to_dict())

profile = (
    FEATURE_SETS[PRIMARY]
    .assign(cluster=labels)
    .groupby('cluster')
    .mean()
    .T
)
profile['spread'] = profile.max(axis=1) - profile.min(axis=1)

print('\nTop 20 most cluster-differentiating variables (original scales):')
print(profile.sort_values('spread', ascending=False).head(20).round(2).to_string())

# Any cluster smaller than ~8 people cannot support percentage claims. Check the sizes
# above before writing a single sentence about a cluster.

## 8. How to report whatever comes out

**If Section 6 found nothing above the null** (the likely outcome):

> *We looked for latent subgroups among the 49 respondents who consented to both survey
> sections, using PCA, t-SNE and UMAP over all encoded variables, and k-means for
> k = 2-8. Silhouette scores for the observed data fell within the distribution obtained
> from 200 datasets in which each variable was independently permuted, so we found no
> subgroup structure distinguishable from chance at this sample size. The t-SNE and UMAP
> projections show apparent groupings, but equally strong groupings appear in the
> permuted data, so we do not interpret them.*

That paragraph is a genuinely good result: it is a correct negative with the evidence
attached, and it demonstrates the methods were understood rather than just run.

**If something did clear the null:** report the silhouette table, the cluster sizes, the
profile from Section 7, and whether it survived the Section 6 feature-set robustness
check. Use the PCA panel coloured by observed variables -- if a cluster corresponds to
something nameable like language preference, say so directly and skip the clustering
machinery; a crosstab of two variables is clearer than an embedding and easier to trust.

**For the write-up either way:** PC1+PC2 variance explained, n=49 and why (dual consent),
the scaling decision, and the fact that no causal reading is available.

### Hazards to not fall into

- Reporting a t-SNE or UMAP plot without the null panel beside it. The plot alone
  overstates the evidence, and it is the mistake these methods are known for.
- Tuning perplexity or `n_neighbors` until a nice picture appears, then reporting that
  setting. Report the grid.
- Interpreting distances between t-SNE/UMAP blobs, or blob sizes. Neither is meaningful.
- Naming clusters with fewer than ~8 members.
- Treating a cluster as a type of student. These are 49 statistics majors in one class,
  89% seniors, 94% the same primary major -- an unusually homogeneous group, which is
  itself part of why structure is hard to find here.

### Where this sits in the 5-way split

This is the exploratory/methods slice (person 1 or 2 in the split at the end of
`scratchwork-clark.ipynb`). It pairs naturally with the preparation-index work, since
both ask "are there meaningfully different kinds of student in this class" -- one by
constructing a measure, the other by letting the data speak. Expect at most **one**
figure and one paragraph from this notebook to reach the report; its main value is
knowing what *not* to claim.